#Guardrail 

PII Middleware 
Human in the loop 
Before the hook 
Afte the LLM 
Layered Guardrails 

Strategies that we hav efor builtin PPI 

Redacted 
Block 
Mask 
Hash

In [1]:
import os 
import dotenv 
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool 


load_dotenv()

@tool 
def customer_lookup(query: str) -> str:
    """Look up customer information"""
    return f"Customer record found for: {query}"


llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.1-8b-instant", 

)


model = create_agent(
    model = llm,
    tools =[customer_lookup],
    middleware =[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input =True,
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,    

        ),
        PIIMiddleware(
            "api_key",
            detector = "sk-[a-zA-Z0-9]{32}",
            strategy ="block",
            apply_to_input=True,
        )
    ]
)


print("Agent with PII middleware successfully!!!! ")





C:\Users\Siddhanth\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Agent with PII middleware successfully!!!! 


In [ ]:
result = model.invoke({
    "messages" : [{
        "role": "user",
        "content" : "My email is john.doe@gmail.com. Can you help me ?"
    }]
})

result



{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL]. Can you help me ?', additional_kwargs={}, response_metadata={}, id='d6efcfba-c1da-42db-91de-70bbfa55d643'),
  AIMessage(content="I can't assist with that request. If you'd like to look up customer information, you could use the following function.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 223, 'total_tokens': 249, 'completion_time': 0.075775488, 'completion_tokens_details': None, 'prompt_time': 0.017695256, 'prompt_tokens_details': None, 'queue_time': 0.0502551, 'total_time': 0.093470744}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019eff24-262f-7423-8d34-0921c33f36c3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 223, 'output_tokens': 26, 'total_tokens': 249})]}

Human in the loop middleware

In [ ]:
from google import genai

# ----------------------------
# Configure Gemini
# ----------------------------
client = genai.Client(api_key="GEMINI_API_KEY")

# ----------------------------
# Guardrail
# ----------------------------
SENSITIVE_KEYWORDS = [
    "delete",
    "transfer money",
    "wire",
    "password",
    "database",
    "shutdown",
    "production"
]


def requires_human_review(prompt: str) -> bool:
    prompt = prompt.lower()

    for keyword in SENSITIVE_KEYWORDS:
        if keyword in prompt:
            return True

    return False


# ----------------------------
# Human Approval
# ----------------------------
def get_human_approval(prompt: str) -> bool:
    print("\n⚠️ Human Review Required")
    print(f"Prompt: {prompt}")

    approval = input("Approve? (yes/no): ").strip().lower()

    return approval == "yes"


# ----------------------------
# Gemini Agent
# ----------------------------
def ask_gemini(prompt: str):
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
    )

    return response.text


# ----------------------------
# Main Loop
# ----------------------------
while True:
    user_prompt = input("\nYou: ")

    if user_prompt.lower() == "exit":
        break

    if requires_human_review(user_prompt):
        approved = get_human_approval(user_prompt)

        if not approved:
            print("❌ Request rejected by human.")
            continue

    answer = ask_gemini(user_prompt)
    print("\nGemini:", answer)